
# 🏷️ Decision Tree Classification


![image](https://miro.medium.com/v2/resize:fit:640/format:webp/1*wj0YGKlKDu0nR50xhaBriw.png)

---

## 🎯 Objectius
- Aquest notebook pretén mostrar un exemple senzill d'ús d'un model KNN - K Nearest Neighbors amb dades sintetitzades.
- Hi ha un exemple utilitzant Gini Índex i Croos-Entropy com a mesura d'impuresa


## 🟦 1. Load Data

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report

# 1. Dataset categòric
data = {
    "color_web": [
        "blau", "vermell", "verd", "blau", "blau", "vermell", "verd", "vermell",
        "blau", "verd", "vermell", "blau", "verd", "blau", "vermell"
    ],
    "dispositiu": [
        "mòbil", "ordinador", "mòbil", "tauleta", "ordinador",
        "mòbil", "tauleta", "ordinador", "mòbil", "ordinador",
        "tauleta", "mòbil", "tauleta", "ordinador", "mòbil"
    ],
    "metode_pagament": [
        "targeta", "paypal", "targeta", "transferència", "targeta",
        "paypal", "transferència", "targeta", "targeta", "paypal",
        "transferència", "paypal", "transferència", "targeta", "paypal"
    ],
    "compra": [
        "sí", "no", "sí", "no", "sí",
        "no", "no", "sí", "sí", "no",
        "no", "sí", "no", "sí", "no"
    ]
}

df = pd.DataFrame(data)

df.head()

,color_web,dispositiu,metode_pagament,compra
0,blau,mòbil,targeta,sí
1,vermell,ordinador,paypal,no
2,verd,mòbil,targeta,sí
3,blau,tauleta,transferència,no
4,blau,ordinador,targeta,sí


## 🟦 2. Preparació de les dades i entrenament

In [ ]:
from sklearn.preprocessing import LabelEncoder

# Variables
X = df[["color_web", "dispositiu", "metode_pagament"]]
y = df["compra"]


le = LabelEncoder()
y_encoded = le.fit_transform(y)    # converteix 'sí'/'no' a enters

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.3, random_state=42, stratify=y_encoded
)

# Preprocessat categòric
cat_features = ["color_web", "dispositiu", "metode_pagament"]

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_features)
    ]
)

# Model KNN dins d’un pipeline
pipeline = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        ("knn", KNeighborsClassifier())
    ]
)

# 2. Definim el GridSearchCV
param_grid = {
    "knn__n_neighbors": [1, 3, 5],     # provar K
    "knn__metric": ["hamming", "manhattan"]  # comparar distàncies
}

grid = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=5,
    scoring="f1"
)

# Entrenem GridSearch
grid.fit(X_train, y_train)

print("Millors hiperparàmetres:", grid.best_params_)
print("Millor score:", grid.best_score_)





Millors hiperparàmetres: {'knn__metric': 'manhattan', 'knn__n_neighbors': 1}
Millor score: 1.0


## 🟦 3. Avaluació

In [ ]:
# Avaluació sobre el conjunt de test
y_pred = grid.predict(X_test)
print("\nInforme de classificació sobre TEST:")
print(classification_report(y_test, y_pred, target_names=le.classes_))



Informe de classificació sobre TEST:
              precision    recall  f1-score   support

          no       1.00      1.00      1.00         3
          sí       1.00      1.00      1.00         2

    accuracy                           1.00         5
   macro avg       1.00      1.00      1.00         5
weighted avg       1.00      1.00      1.00         5

